[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/apartsin/DLCourseHIT/blob/main/notebooks/week11.ipynb)

# Week 11: LSTMs, GRUs & Sequence Tasks
**Introduction to Deep Learning (HIT)** &middot; Part III: Architectures & Representation Learning

Gated recurrent units; how gates restore gradient flow; LSTM versus GRU; sequence classification and sequence-to-sequence tasks.

**Instructor practice notebook.** Run these live demonstrations during the 2-hour practice lesson. The student homework is the weekly lab.

## Goals for the practice lesson

- Build an LSTM or GRU and compare it to the plain RNN.
- Understand how gates restore gradient flow.
- Apply gated networks to a sequence task.

## Setup
Run this first. On Colab, set Runtime > Change runtime type > GPU for the later weeks.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(0)
print('PyTorch', torch.__version__, '| device:', device)

## Demonstration 1
Swap the RNN for an LSTM or GRU on the same task and compare.

In [ ]:
# Same input through RNN, LSTM, and GRU (note the parameter counts)
x = torch.randn(8, 20, 1)
for name, layer in [("RNN", nn.RNN(1, 16, batch_first=True)),
                    ("LSTM", nn.LSTM(1, 16, batch_first=True)),
                    ("GRU", nn.GRU(1, 16, batch_first=True))]:
    out = layer(x)[0]
    print(f"{name:4s}: output {tuple(out.shape)}, params {sum(p.numel() for p in layer.parameters())}")

## Demonstration 2
Walk through the gates and the cell state on the board and in code.

In [ ]:
# The LSTM carries a hidden state h and a cell state c; gates are 4x hidden
lstm = nn.LSTM(1, 4, batch_first=True)
out, (h, c) = lstm(torch.randn(1, 5, 1))
print("hidden h:", tuple(h.shape), "| cell c:", tuple(c.shape))
print("weight_ih_l0:", tuple(lstm.weight_ih_l0.shape), "= (4*hidden, input): input/forget/cell/output gates")

## Demonstration 3
Show behavior on long versus short sequences.

In [ ]:
# Gradient reaching the first step stays larger on long sequences than a plain RNN
lstm = nn.LSTM(1, 8, batch_first=True)
for T in [5, 100]:
    x = torch.randn(2, T, 1, requires_grad=True)
    out, _ = lstm(x); out[:, -1].sum().backward()
    print(f"seq len {T:3d}: gradient to first step ~ {x.grad.abs()[:, 0].mean().item():.2e}")

---
Student materials for this week: the lab handout (`labs/week11.html`) and the curated references (`references/week11.html`) in the course site.